# Notebook 03 | Faithfulness Metrics

This notebook computes global and subgroup-disaggregated
faithfulness metrics (PGI, PGU) for SHAP and LIME, then the three novel disparity metrics
(WSF, FD, EDI) that are the core contribution of the project.

**Run 02a and 02b first**: this notebook consumes their outputs and does not regenerate them.

**Inputs** (from Drive):
- `model_rf.pkl` | trained Random Forest (needed to compute f(x) for PGI/PGU)
- `feature_names.pkl` | list of 14 feature names, in model-input order
- `X_sample.csv` | 500 evaluation instances, shape (500, 14)
- `y_sample.csv` | true labels for the 500 instances
- `sens_sample.csv` | sensitive attributes (sex_raw, race_raw, age_group)
- `shap_values.npy` | SHAP values for the positive class, shape (500, 14)
- `lime_values.npy` | LIME local weights for the positive class, shape (500, 14)

**Outputs** (saved to Drive):
- `metrics_global.csv` | global PGI/PGU baseline for SHAP and LIME
- `metrics_disaggregated.csv` | per metric x explainer x attribute x group score, with group size n
- `metrics_disparity.csv` | WSF, FD, EDI for every (metric, explainer, attribute)
- `global_pgi_pgu.png` | global PGI/PGU, SHAP vs LIME
- `subgroup_sex.png`, `subgroup_race.png`, `subgroup_age_group.png` | per-subgroup PGI/PGU
- `disparity_heatmap.png` | FD and EDI heatmaps across attributes and explainers

## Step 1 | Import Libraries

Note: `pip install` is not needed since only numpy, pandas, matplotlib and scikit-learn libraries are used, and all four are pre-installed on Colab

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import matplotlib.pyplot as plt
from google.colab import drive

print(f"numpy {np.__version__}, pandas {pd.__version__}")

## Step 2 | Mount Drive and Load Artifacts

We load the seven artifacts produced by the notebooks 1, 2a and 2b. The model is loaded together with `feature_names.pkl` because it is required to compute f(x) for PGI/PGU, exactly as 2a and 2b already load it

In [ ]:
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/XAIP/data/'
print(f"Drive mounted > base path: {BASE}")

In [ ]:
#model and feature names (the model is needed to compute f(x) for PGI/PGU) 
with open(BASE + 'model_rf.pkl', 'rb') as f:
    model = pickle.load(f)
with open(BASE + 'feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

# Evalaution sample 
X_sample    = pd.read_csv(BASE + 'X_sample.csv')
y_sample    = pd.read_csv(BASE + 'y_sample.csv').squeeze('columns')
sens_sample = pd.read_csv(BASE + 'sens_sample.csv')

# Reorder feature columns to the model input order, then to a float matrix
X_sample = X_sample[feature_names]
X_mat    = X_sample.to_numpy(dtype=float)

#Attribution matrices (positive class)
shap_values = np.load(BASE + 'shap_values.npy')
lime_values = np.load(BASE + 'lime_values.npy')

print(f"OK model_rf.pkl loaded")
print(f"OK feature_names.pkl loaded ({len(feature_names)} features)")
print(f"OK X_sample loaded        {X_sample.shape}")
print(f"OK y_sample loaded        {y_sample.shape}")
print(f"OK sens_sample loaded     {sens_sample.shape}")
print(f"OK shap_values loaded     {shap_values.shape}")
print(f"OK lime_values loaded     {lime_values.shape}")

In [ ]:
# Every row-aligned object must share the same length and ordering
assert len(X_sample) == len(y_sample) == len(sens_sample) \
       == len(shap_values) == len(lime_values), "Row-count mismatch across artifacts"
assert shap_values.shape == X_mat.shape, f"SHAP shape {shap_values.shape} != X {X_mat.shape}"
assert lime_values.shape == X_mat.shape, f"LIME shape {lime_values.shape} != X {X_mat.shape}"
assert list(sens_sample.columns) == ['sex_raw', 'race_raw', 'age_group'], \
    f"Unexpected sensitive columns: {list(sens_sample.columns)}"

N_SAMPLE = len(X_sample)
print(f"OK Alignment checks passed: {N_SAMPLE} instances, {len(feature_names)} features")

## Step 3 | Faithfulness Metrics: PGI and PGU (global baseline)

**PGI (Prediction Gap on Important features)** masks the `TOP_K` features an explanation ranks
as most important (by `|phi|`) and measures how far the prediction moves. Higher PGI means the
explanation pointed at features that genuinely drive the model, so it is more faithful

**PGU (Prediction Gap on Unimportant features)** masks the `BOTTOM_K` least important features.
A faithful explanation should barely move the prediction here, so lower PGU is more faithful

**Masking baseline (explicit design decision).** A masked feature is replaced by its mean computed
over the 500-instance evaluation sample `X_sample` itself, not over the training set. This keeps
the notebook self-contained (no need to reload `adult_train.csv`) and uses the same reference
distribution the metrics are evaluated on. Rankings are computed per explainer,
since SHAP and LIME disagree on feature ordering.

In [ ]:
TOP_K    = 5
BOTTOM_K = 5

# Masking baseline: per-feature mean over the evaluation sample 
feature_means = X_mat.mean(axis=0)


def compute_pgi_pgu(X, phi, model, means, top_k=TOP_K, bottom_k=BOTTOM_K):
    """Per-instance PGI and PGU for a single attribution matrix.

    PGI(x) = f(x) - f(x with its top-k    |phi| features set to their mean)
    PGU(x) = f(x) - f(x with its bottom-k |phi| features set to their mean)

    Rankings are per instance and per explainer, vectorized over all instances
    """
    X   = np.asarray(X, dtype=float)
    phi = np.asarray(phi, dtype=float)
    n   = X.shape[0]

    #wrap in a dataframe so predict_proba sees the training feature names
    def proba(M):
        return model.predict_proba(pd.DataFrame(M, columns=feature_names))[:, 1]

    fx = proba(X)

    # Rank features per instance by absolute attribution (ascending)
    order   = np.argsort(np.abs(phi), axis=1)
    top_idx = order[:, -top_k:]    # most important
    bot_idx = order[:, :bottom_k]  # least important

    rows_top = np.repeat(np.arange(n), top_k)
    rows_bot = np.repeat(np.arange(n), bottom_k)

    X_top = X.copy()
    X_top[rows_top, top_idx.ravel()] = means[top_idx.ravel()]

    X_bot = X.copy()
    X_bot[rows_bot, bot_idx.ravel()] = means[bot_idx.ravel()]

    pgi = fx - proba(X_top)
    pgu = fx - proba(X_bot)
    return pgi, pgu


pgi_shap, pgu_shap = compute_pgi_pgu(X_mat, shap_values, model, feature_means)
pgi_lime, pgu_lime = compute_pgi_pgu(X_mat, lime_values, model, feature_means)

print(f"OK Per-instance PGI/PGU computed for SHAP and LIME (TOP_K={TOP_K}, BOTTOM_K={BOTTOM_K})")
print(f"  SHAP: PGI in [{pgi_shap.min():.4f}, {pgi_shap.max():.4f}], "
      f"PGU in [{pgu_shap.min():.4f}, {pgu_shap.max():.4f}]")
print(f"  LIME: PGI in [{pgi_lime.min():.4f}, {pgi_lime.max():.4f}], "
      f"PGU in [{pgu_lime.min():.4f}, {pgu_lime.max():.4f}]")

In [ ]:
# Global (aggregate) baseline: mean over all 500 instances
global_df = pd.DataFrame([
    {'explainer': 'SHAP', 'PGI': pgi_shap.mean(), 'PGU': pgu_shap.mean()},
    {'explainer': 'LIME', 'PGI': pgi_lime.mean(), 'PGU': pgu_lime.mean()},
]).set_index('explainer')

print("=" * 60)
print("  GLOBAL FAITHFULNESS BASELINE (mean over 500 instances)")
print("=" * 60)
print(global_df.round(4).to_string())
print("\nHigher PGI = more faithful   |   Lower PGU = more faithful")

## Step 4 | Subgroup Disaggregation

We break PGI and PGU down by each sensitive attribute (sex, race, age_group) and explainer.
`score(m, e, g)` is the mean of the per-instance metric over all instances in group `g`. The
result is a tidy table with one row per metric x explainer x attribute x group, carrying the
group sample size `n`.

> **Small-count subgroups.** In the 500-instance sample, `Amer-Indian-Eskimo` and `Other` (race)
> and `65+` (age_group) contain very few instances. Their scores are reported for completeness and
> should be read considering this

In [ ]:
# Ordered subgroup definitions
ATTRIBUTES = {
    'sex':       ('sex_raw',   ['Male', 'Female']),
    'race':      ('race_raw',  ['White', 'Black', 'Asian-Pac-Islander',
                                'Amer-Indian-Eskimo', 'Other']),
    'age_group': ('age_group', ['<25', '25-35', '35-50', '50-65', '65+']),
}

# Per-instance frame: subgroup labels + the four per-instance metric columns
per_inst = pd.DataFrame({
    'sex':       sens_sample['sex_raw'].values,
    'race':      sens_sample['race_raw'].values,
    'age_group': sens_sample['age_group'].values,
    'PGI_SHAP':  pgi_shap,
    'PGU_SHAP':  pgu_shap,
    'PGI_LIME':  pgi_lime,
    'PGU_LIME':  pgu_lime,
})

records = []
for attr, (col, order) in ATTRIBUTES.items():
    for group in order:
        mask = per_inst[attr] == group
        n_g  = int(mask.sum())
        for metric in ['PGI', 'PGU']:
            for expl in ['SHAP', 'LIME']:
                colname = f"{metric}_{expl}"
                score   = per_inst.loc[mask, colname].mean() if n_g > 0 else np.nan
                records.append({'attribute': attr, 'group': group, 'n': n_g,
                                 'metric': metric, 'explainer': expl, 'score': score})

disagg_df = pd.DataFrame(records)

n_groups = disagg_df[['attribute', 'group']].drop_duplicates().shape[0]
print(f"✓ Disaggregated scores computed -- "
      f"{len(disagg_df)} rows = {n_groups} groups x 2 metrics x 2 explainers")

# group sizes (for the small-count scenario)
print("\nSubgroup sample sizes:")
sizes = disagg_df[['attribute', 'group', 'n']].drop_duplicates()
print(sizes.to_string(index=False))

In [ ]:
#Readable pivot: PGI by subgroup, SHAP vs LIME side by side
pivot_pgi = disagg_df[disagg_df['metric'] == 'PGI'].pivot_table(
    index=['attribute', 'group'], columns='explainer', values='score')
pivot_pgu = disagg_df[disagg_df['metric'] == 'PGU'].pivot_table(
    index=['attribute', 'group'], columns='explainer', values='score')

print("PGI by subgroup (SHAP vs LIME):")
print(pivot_pgi.round(4).to_string())
print("\nPGU by subgroup (SHAP vs LIME):")
print(pivot_pgu.round(4).to_string())

## Step 5 | Disparity Metrics: WSF, FD, EDI

For each (metric, explainer, attribute) we compute the three novel disparity metrics exactly as
defined in the project README:

- **WSF (Worst Subgroup Faithfulness)** = `min_g score(m, e, g)`
- **FD (Faithfulness Disparity)** = `max_g score - min_g score`, flagged when above `0.1`
- **EDI (Explanation Desert Index)**: `EDI(g) = global_score - score(g)`, then
  `EDI(m, e, a) = max_g EDI(g)`; we also record which group attains that maximum.

**Reading direction.** The formulas are applied uniformly to both metrics. For PGI (higher = more
faithful), `WSF` is the least-served group and a positive `EDI` marks a group below the global
average. For PGU (lower = more faithful) the direction inverts, so for PGU the disparity `FD` and
the identity of the worst group are the meaningful equity signals, while `WSF` is still reported as
defined for completeness.

In [ ]:
FD_THRESHOLD = 0.1

global_score = {
    ('PGI', 'SHAP'): pgi_shap.mean(), ('PGU', 'SHAP'): pgu_shap.mean(),
    ('PGI', 'LIME'): pgi_lime.mean(), ('PGU', 'LIME'): pgu_lime.mean(),
}

disparity_rows = []
for attr, (col, order) in ATTRIBUTES.items():
    for metric in ['PGI', 'PGU']:
        for expl in ['SHAP', 'LIME']:
            sub = disagg_df[(disagg_df['attribute'] == attr) &
                            (disagg_df['metric'] == metric) &
                            (disagg_df['explainer'] == expl)].dropna(subset=['score'])
            scores = sub.set_index('group')['score']

            wsf         = scores.min()
            worst_group = scores.idxmin()
            fd          = scores.max() - scores.min()

            gscore        = global_score[(metric, expl)]
            edi_per_group = gscore - scores
            edi           = edi_per_group.max()
            edi_group     = edi_per_group.idxmax()

            disparity_rows.append({
                'attribute': attr, 'metric': metric, 'explainer': expl,
                'WSF': wsf, 'WSF_group': worst_group,
                'FD': fd, 'FD_flag': bool(fd > FD_THRESHOLD),
                'EDI': edi, 'EDI_group': edi_group, 'global': gscore,
            })

disparity_df = pd.DataFrame(disparity_rows)

print("=" * 60)
print("  DISPARITY METRICS  (WSF / FD / EDI)")
print("=" * 60)
show_cols = ['attribute', 'metric', 'explainer', 'WSF', 'WSF_group',
             'FD', 'FD_flag', 'EDI', 'EDI_group']
print(disparity_df[show_cols].round(4).to_string(index=False))

flagged = disparity_df[disparity_df['FD_flag']]
print(f"\n{len(flagged)} of {len(disparity_df)} (metric, explainer, attribute) combinations "
      f"exceed the FD > {FD_THRESHOLD} concern threshold.")

## Step 6 | SHAP vs LIME: Which Explainer Is More Equitable?

For each attribute we compare the two explainers on disparity. Smaller `FD` means explanation
quality is spread more evenly across subgroups, so the explainer with the lower `FD` is the more
equitable one on that attribute. `FD` is a spread and is direction-agnostic, so it is a fair basis
for comparison on both PGI and PGU. We also print `WSF` on PGI for context (higher is better: the
worst-served group still receives a more faithful explanation).

In [ ]:
print("=" * 60)
print("  SHAP vs LIME -- SUBGROUP EQUITY COMPARISON")
print("=" * 60)

equity_records = []
for attr in ATTRIBUTES:
    print(f"\nAttribute: {attr}")
    for metric in ['PGI', 'PGU']:
        row_s = disparity_df[(disparity_df.attribute == attr) &
                             (disparity_df.metric == metric) &
                             (disparity_df.explainer == 'SHAP')].iloc[0]
        row_l = disparity_df[(disparity_df.attribute == attr) &
                             (disparity_df.metric == metric) &
                             (disparity_df.explainer == 'LIME')].iloc[0]
        fd_s, fd_l = row_s['FD'], row_l['FD']
        more_equitable = 'SHAP' if fd_s < fd_l else ('LIME' if fd_l < fd_s else 'tie')
        equity_records.append({'attribute': attr, 'metric': metric,
                                'FD_SHAP': fd_s, 'FD_LIME': fd_l,
                                'more_equitable': more_equitable})
        print(f"  {metric}: FD_SHAP={fd_s:.4f}  FD_LIME={fd_l:.4f}  "
              f"-> more equitable: {more_equitable}")

equity_df = pd.DataFrame(equity_records)

print("\n" + "-" * 60)
print("  PER-ATTRIBUTE VERDICT")
print("-" * 60)
for attr in ATTRIBUTES:
    votes  = equity_df[equity_df.attribute == attr]['more_equitable'].tolist()
    shap_v = votes.count('SHAP')
    lime_v = votes.count('LIME')
    if shap_v > lime_v:
        verdict = 'SHAP is more equitable'
    elif lime_v > shap_v:
        verdict = 'LIME is more equitable'
    else:
        verdict = 'SHAP and LIME are comparable'
    print(f"  {attr:10s}: {verdict}  (FD wins -> SHAP:{shap_v}  LIME:{lime_v})")

# Largest single disparity gap between the two explainers
equity_df['gap'] = (equity_df['FD_SHAP'] - equity_df['FD_LIME']).abs()
worst = equity_df.loc[equity_df['gap'].idxmax()]
print(f"\nLargest SHAP/LIME disparity gap: {worst['metric']} on '{worst['attribute']}' "
      f"(|FD_SHAP - FD_LIME| = {worst['gap']:.4f})")

## Step 7 | Visualizations

### 7a > Global PGI/PGU, SHAP vs LIME

In [ ]:
labels = ['PGI', 'PGU']
shap_g = [global_df.loc['SHAP', 'PGI'], global_df.loc['SHAP', 'PGU']]
lime_g = [global_df.loc['LIME', 'PGI'], global_df.loc['LIME', 'PGU']]

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w / 2, shap_g, w, label='SHAP', color='steelblue')
ax.bar(x + w / 2, lime_g, w, label='LIME', color='darkorange')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Mean prediction gap')
ax.set_title('Global Faithfulness -- SHAP vs LIME (n=500)')
ax.legend()
plt.tight_layout()
plt.savefig(BASE + 'global_pgi_pgu.png', dpi=150)
plt.show()

### 7b > Per-Subgroup PGI and PGU (one figure per sensitive attribute)

In [ ]:
for attr, (col, order) in ATTRIBUTES.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric in zip(axes, ['PGI', 'PGU']):
        sub = disagg_df[(disagg_df.attribute == attr) & (disagg_df.metric == metric)]
        piv = sub.pivot_table(index='group', columns='explainer',
                              values='score').reindex(order)
        x = np.arange(len(order))
        w = 0.35
        ax.bar(x - w / 2, piv['SHAP'].values, w, label='SHAP', color='steelblue')
        ax.bar(x + w / 2, piv['LIME'].values, w, label='LIME', color='darkorange')
        ax.axhline(0, color='gray', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(order, rotation=30, ha='right')
        ax.set_ylabel('Mean prediction gap')
        ax.set_title(f'{metric} by {attr}')
        ax.legend()
    plt.suptitle(f'Subgroup Faithfulness -- {attr} (SHAP vs LIME)', fontsize=13)
    plt.tight_layout()
    plt.savefig(BASE + f'subgroup_{attr}.png', dpi=150)
    plt.show()

### 7c > Disparity Heatmaps (FD and EDI)

Rows are (metric, explainer) combinations, columns are sensitive attributes. Higher values (darker cells) mean more disparity, i.e. worse-served subgroups.

In [ ]:
combos     = [('PGI', 'SHAP'), ('PGI', 'LIME'), ('PGU', 'SHAP'), ('PGU', 'LIME')]
row_labels = [f"{m}-{e}" for m, e in combos]
attrs      = list(ATTRIBUTES.keys())

FD_mat  = np.zeros((len(combos), len(attrs)))
EDI_mat = np.zeros((len(combos), len(attrs)))
for i, (m, e) in enumerate(combos):
    for j, a in enumerate(attrs):
        r = disparity_df[(disparity_df.metric == m) &
                         (disparity_df.explainer == e) &
                         (disparity_df.attribute == a)].iloc[0]
        FD_mat[i, j]  = r['FD']
        EDI_mat[i, j] = r['EDI']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mat, name in zip(axes, [FD_mat, EDI_mat], ['FD', 'EDI']):
    im = ax.imshow(mat, cmap='Oranges', aspect='auto')
    ax.set_xticks(range(len(attrs)))
    ax.set_xticklabels(attrs)
    ax.set_yticks(range(len(combos)))
    ax.set_yticklabels(row_labels)
    ax.set_title(f'{name} across attributes')
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat[i, j]:.3f}", ha='center', va='center', fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle('Disparity Heatmaps | FD and EDI (higher = more disparity)', fontsize=13)
plt.tight_layout()
plt.savefig(BASE + 'disparity_heatmap.png', dpi=150)
plt.show()

## Step 8 | Save Results

All result tables are written to Drive as CSV; the figures were already saved as PNG during Step 7

In [ ]:
global_out = global_df.reset_index()
global_out.to_csv(BASE + 'metrics_global.csv', index=False)
disagg_df.to_csv(BASE + 'metrics_disaggregated.csv', index=False)
disparity_df.to_csv(BASE + 'metrics_disparity.csv', index=False)

print(f"OK metrics_global.csv saved")
print(f"OK metrics_disaggregated.csv saved")
print(f"OK metrics_disparity.csv saved")

# Reload verification
_g = pd.read_csv(BASE + 'metrics_global.csv')
_d = pd.read_csv(BASE + 'metrics_disaggregated.csv')
_p = pd.read_csv(BASE + 'metrics_disparity.csv')
assert len(_d) == len(disagg_df) and len(_p) == len(disparity_df)
print(f"OK Reload verification passed")

## Summary

In [ ]:
print("=" * 60)
print("  SUMMARY | OUTPUTS")
print("=" * 60)
print(f"\nBASE PATH:\n  {BASE}")

print(f"\nFILES PRODUCED:")
files = ['metrics_global.csv', 'metrics_disaggregated.csv', 'metrics_disparity.csv',
         'global_pgi_pgu.png', 'subgroup_sex.png', 'subgroup_race.png',
         'subgroup_age_group.png', 'disparity_heatmap.png']
for fname in files:
    fpath = BASE + fname
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fname:32s}  {size_kb:.1f} KB")

print(f"\nGLOBAL BASELINE (mean over {N_SAMPLE} instances):")
print(global_df.round(4).to_string())

print(f"\nFD CONCERN FLAGS (FD > {FD_THRESHOLD}):")
flagged = disparity_df[disparity_df['FD_flag']]
if len(flagged):
    for _, r in flagged.iterrows():
        print(f"  {r['metric']}-{r['explainer']:4s} on {r['attribute']:10s}: "
              f"FD={r['FD']:.4f}  most-underserved group (EDI) -> {r['EDI_group']}")
else:
    print("  none")

print(f"\nOK Notebook 03 complete | faithfulness + disparity metrics done")